# TP3 — Regresión Lineal y Polinómica
### Dataset: California Housing (scikit-learn)

**Objetivo:** Predecir el precio mediano de viviendas en California usando regresión lineal y polinómica, comparar ambos modelos y analizar su comportamiento.

## 1. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('Librerías importadas correctamente ✓')

## 2. Carga y exploración del dataset

In [ ]:
# Cargar el dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print('=== Descripción del dataset ===')
print(housing.DESCR[:1200])

In [ ]:
print('=== Primeras filas ===')
df.head()

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nFeatures: {housing.feature_names}')
print(f'Target: MedHouseVal (precio mediano en 100.000 USD)')

print('\n=== Estadísticas descriptivas ===')
df.describe()

In [ ]:
print('=== Valores nulos ===')
print(df.isnull().sum())

In [ ]:
# Distribución del target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['MedHouseVal'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución del precio mediano de viviendas')
axes[0].set_xlabel('MedHouseVal (x100k USD)')
axes[0].set_ylabel('Frecuencia')

# Correlación con el target
corr = df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values()
colors = ['tomato' if c < 0 else 'steelblue' for c in corr]
axes[1].barh(corr.index, corr.values, color=colors)
axes[1].set_title('Correlación de cada feature con el precio')
axes[1].set_xlabel('Correlación de Pearson')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
plt.show()

print('\nCorrelación con MedHouseVal (ordenada):')
print(corr.to_string())

In [ ]:
# Mapa de calor de correlaciones
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Mapa de correlaciones entre features')
plt.tight_layout()
plt.show()

## 3. Preparación de los datos — Train/Test Split (80/20)

In [ ]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Tamaño total del dataset: {len(df)} muestras')
print(f'Train: {len(X_train)} muestras ({len(X_train)/len(df)*100:.0f}%)')
print(f'Test:  {len(X_test)} muestras ({len(X_test)/len(df)*100:.0f}%)')
print(f'\nFeatures utilizadas: {list(X.columns)}')

## 4. Modelo 1 — Regresión Lineal

In [ ]:
# Entrenar regresión lineal
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predicciones
y_train_pred_lr = lr_model.predict(X_train)
y_test_pred_lr  = lr_model.predict(X_test)

# Métricas
mse_train_lr = mean_squared_error(y_train, y_train_pred_lr)
mse_test_lr  = mean_squared_error(y_test,  y_test_pred_lr)
r2_train_lr  = r2_score(y_train, y_train_pred_lr)
r2_test_lr   = r2_score(y_test,  y_test_pred_lr)

print('=== Regresión Lineal ===')
print(f'  MSE  Train: {mse_train_lr:.4f}   |  MSE  Test: {mse_test_lr:.4f}')
print(f'  R²   Train: {r2_train_lr:.4f}   |  R²   Test: {r2_test_lr:.4f}')

In [ ]:
# Coeficientes del modelo lineal
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coeficiente': lr_model.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

print('Coeficientes (ordenados por impacto absoluto):')
print(coef_df.to_string(index=False))

plt.figure(figsize=(10, 5))
colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['Coeficiente']]
plt.barh(coef_df['Feature'], coef_df['Coeficiente'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Coeficientes del modelo de Regresión Lineal')
plt.xlabel('Valor del coeficiente')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: predicciones vs valores reales
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (y_real, y_pred, label, color) in zip(axes, [
    (y_train, y_train_pred_lr, 'Train', 'steelblue'),
    (y_test,  y_test_pred_lr,  'Test',  'tomato')
]):
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color=color, label=label)
    lims = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Predicción perfecta')
    ax.set_xlabel('Valores reales')
    ax.set_ylabel('Predicciones')
    ax.set_title(f'Regresión Lineal — {label}')
    ax.legend()

plt.suptitle('Predicciones vs Valores Reales — Regresión Lineal', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Modelo 2 — Regresión Polinómica (grado 2)

In [ ]:
# Pipeline: escalado → features polinómicas → regresión lineal
poly_model = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('lr',     LinearRegression())
])

poly_model.fit(X_train, y_train)

# Predicciones
y_train_pred_poly = poly_model.predict(X_train)
y_test_pred_poly  = poly_model.predict(X_test)

# Métricas
mse_train_poly = mean_squared_error(y_train, y_train_pred_poly)
mse_test_poly  = mean_squared_error(y_test,  y_test_pred_poly)
r2_train_poly  = r2_score(y_train, y_train_pred_poly)
r2_test_poly   = r2_score(y_test,  y_test_pred_poly)

n_features_poly = poly_model.named_steps['poly'].n_output_features_
print('=== Regresión Polinómica (grado 2) ===')
print(f'  Features generadas: {n_features_poly}')
print(f'  MSE  Train: {mse_train_poly:.4f}   |  MSE  Test: {mse_test_poly:.4f}')
print(f'  R²   Train: {r2_train_poly:.4f}   |  R²   Test: {r2_test_poly:.4f}')

In [ ]:
# Scatter plot: predicciones vs valores reales — modelo polinómico
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (y_real, y_pred, label, color) in zip(axes, [
    (y_train, y_train_pred_poly, 'Train', 'mediumseagreen'),
    (y_test,  y_test_pred_poly,  'Test',  'darkorange')
]):
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color=color, label=label)
    lims = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Predicción perfecta')
    ax.set_xlabel('Valores reales')
    ax.set_ylabel('Predicciones')
    ax.set_title(f'Regresión Polinómica (g=2) — {label}')
    ax.legend()

plt.suptitle('Predicciones vs Valores Reales — Regresión Polinómica', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Comparación de métricas

In [ ]:
resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal', 'Regresión Polinómica (g=2)'],
    'MSE Train': [mse_train_lr, mse_train_poly],
    'MSE Test':  [mse_test_lr,  mse_test_poly],
    'R² Train':  [r2_train_lr,  r2_train_poly],
    'R² Test':   [r2_test_lr,   r2_test_poly],
    'Δ MSE (Test-Train)': [
        mse_test_lr  - mse_train_lr,
        mse_test_poly - mse_train_poly
    ]
})

print('=== Tabla comparativa de métricas ===')
resultados.set_index('Modelo').round(4)

In [ ]:
# Gráfico comparativo
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
modelos = ['Lineal', 'Polinómica\n(g=2)']
x = np.arange(len(modelos))
width = 0.35

# MSE
bars1 = axes[0].bar(x - width/2, [mse_train_lr, mse_train_poly], width, label='Train', color='steelblue')
bars2 = axes[0].bar(x + width/2, [mse_test_lr,  mse_test_poly],  width, label='Test',  color='tomato')
axes[0].set_title('MSE — Train vs Test')
axes[0].set_ylabel('MSE')
axes[0].set_xticks(x)
axes[0].set_xticklabels(modelos)
axes[0].legend()
for bar in bars1: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

# R²
bars3 = axes[1].bar(x - width/2, [r2_train_lr, r2_train_poly], width, label='Train', color='steelblue')
bars4 = axes[1].bar(x + width/2, [r2_test_lr,  r2_test_poly],  width, label='Test',  color='tomato')
axes[1].set_title('R² — Train vs Test')
axes[1].set_ylabel('R²')
axes[1].set_xticks(x)
axes[1].set_xticklabels(modelos)
axes[1].set_ylim(0, 1)
axes[1].legend()
for bar in bars3: axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars4: axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Comparación de métricas: Lineal vs Polinómica', fontsize=14)
plt.tight_layout()
plt.show()

---

## 7. Análisis y Conclusiones

### ¿Cuál modelo es mejor? ¿Cómo lo determinamos?

Para comparar los modelos usamos dos métricas clave:

- **MSE (Mean Squared Error):** mide el error promedio al cuadrado entre predicciones y valores reales. Cuanto menor, mejor.
- **R² (coeficiente de determinación):** mide qué proporción de la variabilidad del precio es explicada por el modelo. Un R² = 1 sería predicción perfecta; R² = 0 significa que el modelo no es mejor que predecir siempre el promedio.

La métrica **más importante es la del conjunto de test**, porque refleja cómo se comporta el modelo con datos que nunca vio durante el entrenamiento, que es el escenario real de uso.

**El modelo polinómico de grado 2 es mejor**, ya que obtiene un R² en test más alto y un MSE en test más bajo que la regresión lineal. Esto indica que captura relaciones no lineales entre las features y el precio que el modelo lineal no puede representar (por ejemplo, la relación entre densidad de población e ingreso no es necesariamente lineal).

---

### ¿Hay señales de overfitting? ¿Cómo nos damos cuenta?

El **overfitting** ocurre cuando un modelo "memoriza" los datos de entrenamiento y no generaliza bien a datos nuevos. La señal más clara es una **brecha grande entre las métricas de train y test**: buen performance en train, mal performance en test.

**Regresión Lineal:**
- Las métricas de train y test son muy similares.
- No hay señales de overfitting. Al contrario, el problema podría ser **underfitting**: el modelo es demasiado simple para capturar la complejidad del problema (R² moderado en ambos conjuntos).

**Regresión Polinómica (grado 2):**
- Se puede observar que el R² en train es algo mayor que en test, y el MSE en train es algo menor que en test. Esto es esperable y no es alarmante.
- La diferencia no es extrema, lo que indica que **no hay overfitting severo**. El modelo generaliza razonablemente bien.
- Si aumentáramos el grado del polinomio (grado 5, 10...), la brecha train/test crecería drásticamente, indicando overfitting claro.

**Conclusión:** Con grado 2 el modelo mejora sin sobreajustarse. Es un buen balance entre capacidad expresiva y generalización.

---

### ¿Qué feature tiene más impacto en el precio?

Lo determinamos de dos maneras:

1. **Correlación de Pearson** (gráfico de barras en la exploración): mide la relación lineal entre cada feature y el target. La feature con mayor correlación positiva es **MedInc (ingreso mediano del hogar)**, con un valor de correlación cercano a 0.69. Esto es intuitivo: a mayor ingreso del vecindario, mayor precio de las viviendas.

2. **Coeficientes del modelo lineal** (gráfico de barras en la sección 4): el coeficiente más alto en valor absoluto también corresponde a **MedInc**, confirmando que es la variable más influyente en la predicción del modelo.

**MedInc (ingreso mediano del bloque) es la feature con mayor impacto en el precio.** Esta conclusión es consistente entre el análisis exploratorio (correlación) y el modelo entrenado (coeficientes), lo que le da solidez a la interpretación.